In [3]:
import pandas as pd
import numpy as np
#from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier

# Load the dataset
df = pd.read_csv('./DATA/application_train.csv')
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
df.shape


(307511, 122)

In [5]:
selected_features = [
    # numeric
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED",
    "DAYS_BIRTH",
    "CNT_FAM_MEMBERS",
    "CNT_CHILDREN",
    
    # categorical
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_FAMILY_STATUS"
]

# Mantén solo lo necesario
use_cols = ["TARGET"] + selected_features
df_small = df[use_cols].copy()

df_small.shape, df_small["TARGET"].value_counts(normalize=True)

((307511, 13),
 TARGET
 0    0.919271
 1    0.080729
 Name: proportion, dtype: float64)

In [6]:
df_small.head()

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_BIRTH,CNT_FAM_MEMBERS,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,OCCUPATION_TYPE,NAME_FAMILY_STATUS
0,1,202500.0,406597.5,24700.5,351000.0,-637,-9461,1.0,0,Working,Secondary / secondary special,Laborers,Single / not married
1,0,270000.0,1293502.5,35698.5,1129500.0,-1188,-16765,2.0,0,State servant,Higher education,Core staff,Married
2,0,67500.0,135000.0,6750.0,135000.0,-225,-19046,1.0,0,Working,Secondary / secondary special,Laborers,Single / not married
3,0,135000.0,312682.5,29686.5,297000.0,-3039,-19005,2.0,0,Working,Secondary / secondary special,Laborers,Civil marriage
4,0,121500.0,513000.0,21865.5,513000.0,-3038,-19932,1.0,0,Working,Secondary / secondary special,Core staff,Single / not married


In [7]:
y = df_small["TARGET"]
X = df_small.drop(columns=["TARGET"])

X.shape, y.value_counts(normalize=True)

((307511, 12),
 TARGET
 0    0.919271
 1    0.080729
 Name: proportion, dtype: float64)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((246008, 12), (61503, 12))

In [10]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

clf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [11]:
clf.fit(X_train, y_train)
print("Model trained successfully!")

/opt/anaconda3/envs/prototyping/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Model trained successfully!


In [12]:
from sklearn.metrics import roc_auc_score, classification_report

# Prob default
y_prob = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print("ROC AUC:", round(auc, 4))

# Threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, digits=3))

ROC AUC: 0.6488
              precision    recall  f1-score   support

           0      0.948     0.597     0.732     56538
           1      0.120     0.626     0.201      4965

    accuracy                          0.599     61503
   macro avg      0.534     0.611     0.467     61503
weighted avg      0.881     0.599     0.690     61503



In [13]:
import joblib

joblib.dump(
    {
        "model": clf,
        "selected_features": selected_features
    },
    "Models/microcredit_model.joblib"
)

print("Saved → Models/microcredit_model.joblib")

Saved → Models/microcredit_model.joblib


In [14]:
metadata = {
    "model_type": "LogisticRegression",
    "auc": 0.6488,
    "features": selected_features,
    "note": "Baseline interpretable model for microcredit showcase"
}

joblib.dump(metadata, "Models/model_metadata.joblib")

['Models/model_metadata.joblib']

In [15]:
def risk_to_eligibility_score(risk_prob: float) -> float:
    "Converts probability of default into a 0–100 score (100 = best)."

    return round(100 * (1 - risk_prob), 1)

# sanity check
risk_to_eligibility_score(0.2)

80.0

In [17]:
# First user test
sample = X_test.iloc[[5]]

risk_prob = clf.predict_proba(sample)[0, 1]
eligibility_score = risk_to_eligibility_score(risk_prob)

risk_prob, eligibility_score

(np.float64(0.5639283188039697), np.float64(43.6))

In [15]:
import numpy as np
import pandas as pd

# accessing the trained model
logreg = clf.named_steps["model"]
preprocessor = clf.named_steps["preprocess"]

# feature names after preprocessing
feature_names = (
    preprocessor.named_transformers_["num"]
    .get_feature_names_out()
    .tolist()
    + preprocessor.named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(cat_cols)
    .tolist()
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": logreg.coef_[0]
}).sort_values("coef", key=abs, ascending=False)

coef_df.head(10)

,feature,coef
11,NAME_INCOME_TYPE_Pensioner,-14.814987
4,DAYS_EMPLOYED,7.504359
15,NAME_INCOME_TYPE_Working,4.938480
9,NAME_INCOME_TYPE_Commercial associate,4.757777
12,NAME_INCOME_TYPE_State servant,4.681866
3,AMT_GOODS_PRICE,-1.334726
1,AMT_CREDIT,1.167499
16,NAME_EDUCATION_TYPE_Academic degree,-1.116904
13,NAME_INCOME_TYPE_Student,-0.623545
30,OCCUPATION_TYPE_Low-skill Laborers,0.610216


## Model Feature Interpretation (Informal Economy Context)

To simulate credit assessment in an informal economy setting, we intentionally restrict the model to a subset of variables that can be realistically interpreted as behavioral or socio-economic signals observable in a mobile money environment.

The selected features are interpreted as follows:

---

### Income & Financial Capacity

- **AMT_INCOME_TOTAL**  
  Interpreted as estimated monthly or annual income level. In informal contexts, this represents self-declared or transaction-inferred income rather than payroll-based earnings.

- **AMT_CREDIT**  
  The requested credit amount, reflecting the financial needs or borrowing intent of the individual.

- **AMT_ANNUITY**  
  Proxy for repayment burden relative to income.

- **AMT_GOODS_PRICE**  
  Approximate value of financed goods, used as an indicator of financial scale and consumption level.

---

### Stability & Economic Regularity

- **DAYS_EMPLOYED**  
  Reinterpreted as a proxy for income stability over time rather than formal employment duration. Longer observed stability suggests lower volatility in earnings.

- **DAYS_BIRTH**  
  Used as a demographic stability indicator, as life stage can correlate with financial risk behavior.

---

### Household Context

- **CNT_FAM_MEMBERS**  
  Represents household financial burden and dependency ratio.

- **CNT_CHILDREN**  
  Additional proxy for financial obligations.

---

### Income Pattern Proxies (Behavioral Interpretation)

- **NAME_INCOME_TYPE**  
  Not interpreted as formal employment classification. Instead, this variable is treated as a proxy for income regularity patterns (e.g., stable vs. irregular income sources).

- **OCCUPATION_TYPE**  
  Used as a signal of economic activity type rather than official job category.

- **NAME_EDUCATION_TYPE**  
  Proxy for financial literacy and long-term earning potential.

- **NAME_FAMILY_STATUS**  
  Contextual variable reflecting household structure and financial responsibility.

---

## Conceptual Framing

Although some variables originate from a formal credit dataset, they are reinterpreted within an informal finance narrative. The model does not assume access to formal payroll records or tax documentation. Instead, these features are treated as proxies capturing:

- Income level  
- Income stability  
- Household burden  
- Behavioral financial signals  

This approach aligns with real-world mobile money and alternative credit systems, which rely on indirect indicators rather than formal financial documentation.